In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

In [ ]:
CASE_INDEX = 4

In [ ]:
PROJECT_ROOT = Path.cwd().parent.parent.parent

EXTRACTED_DIR = (
        PROJECT_ROOT
        / "data"
        / "extracted"
        / "merged_cells"
)

MANIFEST_PATH = EXTRACTED_DIR / "manifest.csv"

ROLE_NAMES = (
    "cell_1",
    "cell_2",
    "merged_cell",
)

REQUIRED_ARTIFACTS = (
    "raw",
    "preprocessed",
    "binary_mask",
    "instance_labels",
    "selected_cell_mask",
)

print("Project root:")
print(PROJECT_ROOT)

print("\nExtracted cases:")
print(EXTRACTED_DIR)

print("\nManifest:")
print(MANIFEST_PATH)

In [ ]:
def discover_merged_cell_cases(
        extracted_dir: Path,
        manifest_path: Path,
) -> pd.DataFrame:
    if not manifest_path.exists():
        raise FileNotFoundError(
            f"Merged-cell manifest was not found:\n{manifest_path}"
        )

    cases = pd.read_csv(manifest_path)

    required_columns = {
        "case_name",
        "sample_id",
        "cell_1_frame",
        "cell_1_id",
        "cell_2_frame",
        "cell_2_id",
        "merged_frame",
        "merged_cell_id",
    }

    missing_columns = required_columns.difference(cases.columns)

    if missing_columns:
        raise KeyError(
            "The merged-cell manifest is missing required columns: "
            f"{sorted(missing_columns)}"
        )

    cases = cases.copy()

    cases["case_dir"] = cases["case_name"].map(
        lambda name: extracted_dir / str(name)
    )

    cases["case_exists"] = cases["case_dir"].map(Path.exists)

    for role in ROLE_NAMES:
        cases[f"has_{role}"] = cases["case_dir"].map(
            lambda path, role=role: (path / role).is_dir()
        )

    return cases.sort_values(
        [
            "sample_id",
            "merged_frame",
            "merged_cell_id",
        ]
    ).reset_index(drop=True)


cases = discover_merged_cell_cases(
    extracted_dir=EXTRACTED_DIR,
    manifest_path=MANIFEST_PATH,
)

cases[
    [
        "case_name",
        "sample_id",
        "cell_1_frame",
        "cell_1_id",
        "cell_2_frame",
        "cell_2_id",
        "merged_frame",
        "merged_cell_id",
        "has_cell_1",
        "has_cell_2",
        "has_merged_cell",
    ]
]

In [ ]:
if not 0 <= CASE_INDEX < len(cases):
    raise IndexError(
        f"CASE_INDEX must be between 0 and {len(cases) - 1}; "
        f"received {CASE_INDEX}."
    )

selected_case = cases.iloc[CASE_INDEX]

CASE_NAME = str(selected_case["case_name"])
CASE_DIR = Path(selected_case["case_dir"])

print("Selected case:")
print(CASE_NAME)

print("\nCase directory:")
print(CASE_DIR)

print(
    "\nParent 1:",
    f"frame={int(selected_case['cell_1_frame'])},",
    f"cell_id={int(selected_case['cell_1_id'])}",
)

print(
    "Parent 2:",
    f"frame={int(selected_case['cell_2_frame'])},",
    f"cell_id={int(selected_case['cell_2_id'])}",
)

print(
    "Merged cell:",
    f"frame={int(selected_case['merged_frame'])},",
    f"cell_id={int(selected_case['merged_cell_id'])}",
)

In [ ]:
def load_role(
        case_dir: Path,
        role: str,
) -> dict[str, Any]:
    if role not in ROLE_NAMES:
        raise ValueError(
            f"Unknown role {role!r}. Expected one of {ROLE_NAMES}."
        )

    role_dir = case_dir / role
    metadata_path = role_dir / "metadata.json"
    features_path = role_dir / "features.csv"

    if not role_dir.is_dir():
        raise FileNotFoundError(
            f"Role directory was not found:\n{role_dir}"
        )

    if not metadata_path.exists():
        raise FileNotFoundError(
            f"Role metadata was not found:\n{metadata_path}"
        )

    with metadata_path.open("r", encoding="utf-8") as file:
        metadata = json.load(file)

    artifact_files = metadata.get("artifact_files", {})

    arrays: dict[str, np.ndarray] = {}
    artifact_paths: dict[str, Path] = {}

    for artifact_name in REQUIRED_ARTIFACTS:
        filename = artifact_files.get(
            artifact_name,
            f"{artifact_name}.npy",
        )

        artifact_path = role_dir / filename

        if not artifact_path.exists():
            raise FileNotFoundError(
                f"Required artifact {artifact_name!r} is missing "
                f"for role {role!r}:\n{artifact_path}"
            )

        arrays[artifact_name] = np.load(
            artifact_path,
            allow_pickle=False,
        )

        artifact_paths[artifact_name] = artifact_path

    features = None

    if features_path.exists():
        features = pd.read_csv(features_path)

    shapes = {
        name: array.shape
        for name, array in arrays.items()
    }

    if len(set(shapes.values())) != 1:
        raise ValueError(
            f"Artifacts for role {role!r} are not voxel-aligned: "
            f"{shapes}"
        )

    selected_mask = arrays["selected_cell_mask"].astype(bool)

    if not selected_mask.any():
        raise ValueError(
            f"The selected-cell mask for role {role!r} is empty."
        )

    return {
        "role": role,
        "directory": role_dir,
        "metadata_path": metadata_path,
        "metadata": metadata,
        "features": features,
        "artifact_paths": artifact_paths,
        "arrays": arrays,
        "shape": next(iter(shapes.values())),
        "frame": int(metadata["frame"]),
        "cell_id": int(metadata["cell_id"]),
    }

In [ ]:
with (CASE_DIR / "metadata.json").open(
        "r",
        encoding="utf-8",
) as file:
    case_metadata = json.load(file)

case_data = {
    role: load_role(
        case_dir=CASE_DIR,
        role=role,
    )
    for role in ROLE_NAMES
}

cell_1 = case_data["cell_1"]
cell_2 = case_data["cell_2"]
merged_cell = case_data["merged_cell"]

In [ ]:
role_summary = pd.DataFrame(
    [
        {
            "role": role,
            "frame": data["frame"],
            "cell_id": data["cell_id"],
            "shape": data["shape"],
            "selected_voxels": int(
                data["arrays"]["selected_cell_mask"].sum()
            ),
            "raw_dtype": str(
                data["arrays"]["raw"].dtype
            ),
            "preprocessed_dtype": str(
                data["arrays"]["preprocessed"].dtype
            ),
        }
        for role, data in case_data.items()
    ]
)

role_summary

In [ ]:
role_shapes = {
    role: data["shape"]
    for role, data in case_data.items()
}

if len(set(role_shapes.values())) != 1:
    raise ValueError(
        "The three role crops must have the same shape. "
        f"Found: {role_shapes}"
    )

if cell_1["frame"] >= merged_cell["frame"]:
    raise ValueError(
        "Cell 1 must occur before the merged-cell frame."
    )

if cell_2["frame"] >= merged_cell["frame"]:
    raise ValueError(
        "Cell 2 must occur before the merged-cell frame."
    )

print("Complete case loaded successfully.")
print("Shared crop shape:", merged_cell["shape"])
print("Cell 1 frame:", cell_1["frame"])
print("Cell 2 frame:", cell_2["frame"])
print("Merged frame:", merged_cell["frame"])

In [ ]:
cell_1_raw = cell_1["arrays"]["raw"]
cell_1_preprocessed = cell_1["arrays"]["preprocessed"]
cell_1_mask = cell_1["arrays"]["selected_cell_mask"].astype(bool)

cell_2_raw = cell_2["arrays"]["raw"]
cell_2_preprocessed = cell_2["arrays"]["preprocessed"]
cell_2_mask = cell_2["arrays"]["selected_cell_mask"].astype(bool)

merged_raw = merged_cell["arrays"]["raw"]
merged_preprocessed = merged_cell["arrays"]["preprocessed"]
merged_mask = merged_cell["arrays"]["selected_cell_mask"].astype(bool)

Split analysis

In [ ]:
# ============================================================
# 1. Physical voxel dimensions
# ============================================================

DEFAULT_VOXEL_SIZE_ZYX = np.asarray(
    [1.625, 0.40625, 0.40625],
    dtype=float,
)


def resolve_voxel_size_zyx(
        case_metadata: dict,
        role_metadata: dict,
) -> np.ndarray:
    candidate_values = (
        role_metadata.get("voxel_size_zyx"),
        case_metadata.get("voxel_size_zyx"),
    )

    for value in candidate_values:
        if value is None:
            continue

        voxel_size = np.asarray(value, dtype=float)

        if voxel_size.shape == (3,):
            return voxel_size

    return DEFAULT_VOXEL_SIZE_ZYX.copy()


VOXEL_SIZE_ZYX = resolve_voxel_size_zyx(
    case_metadata=case_metadata,
    role_metadata=merged_cell["metadata"],
)

VOXEL_VOLUME_UM3 = float(np.prod(VOXEL_SIZE_ZYX))

print("Voxel size (z, y, x):", VOXEL_SIZE_ZYX)
print("Voxel volume:", VOXEL_VOLUME_UM3, "µm³")

In [ ]:
# ============================================================
# 2. Translation-independent cell-shape properties
# ============================================================

def describe_cell_mask(
        mask: np.ndarray,
        voxel_size_zyx: np.ndarray,
) -> dict:
    mask = np.asarray(mask, dtype=bool)

    coordinates_zyx = np.argwhere(mask)

    if len(coordinates_zyx) < 4:
        raise ValueError(
            "The selected-cell mask does not contain enough voxels "
            "for shape analysis."
        )

    physical_coordinates = (
            coordinates_zyx.astype(float)
            * voxel_size_zyx[None, :]
    )

    centroid_physical = physical_coordinates.mean(axis=0)
    centered_coordinates = (
            physical_coordinates
            - centroid_physical[None, :]
    )

    covariance = np.cov(
        centered_coordinates,
        rowvar=False,
        bias=True,
    )

    eigenvalues, eigenvectors = np.linalg.eigh(covariance)

    descending_order = np.argsort(eigenvalues)[::-1]

    eigenvalues = eigenvalues[descending_order]
    eigenvectors = eigenvectors[:, descending_order]

    projected_coordinates = (
            centered_coordinates
            @ eigenvectors
    )

    lower_percentile = np.percentile(
        projected_coordinates,
        2.5,
        axis=0,
    )

    upper_percentile = np.percentile(
        projected_coordinates,
        97.5,
        axis=0,
    )

    principal_extents_um = (
            upper_percentile
            - lower_percentile
    )

    major_axis_um = float(principal_extents_um[0])
    middle_axis_um = float(principal_extents_um[1])
    minor_axis_um = float(principal_extents_um[2])

    voxel_count = int(mask.sum())
    volume_um3 = float(
        voxel_count
        * np.prod(voxel_size_zyx)
    )

    equivalent_radius_um = float(
        (
                3.0
                * volume_um3
                / (4.0 * np.pi)
        )
        ** (1.0 / 3.0)
    )

    minimum_coordinates = coordinates_zyx.min(axis=0)
    maximum_coordinates = coordinates_zyx.max(axis=0)

    bbox_shape_voxels = (
            maximum_coordinates
            - minimum_coordinates
            + 1
    )

    bbox_dimensions_um = (
            bbox_shape_voxels
            * voxel_size_zyx
    )

    bbox_voxel_count = int(
        np.prod(bbox_shape_voxels)
    )

    extent = (
        voxel_count / bbox_voxel_count
        if bbox_voxel_count > 0
        else np.nan
    )

    elongation = (
        major_axis_um / minor_axis_um
        if minor_axis_um > 0
        else np.inf
    )

    flatness = (
        middle_axis_um / minor_axis_um
        if minor_axis_um > 0
        else np.inf
    )

    return {
        "voxel_count": voxel_count,
        "volume_um3": volume_um3,
        "equivalent_radius_um": equivalent_radius_um,
        "major_axis_um": major_axis_um,
        "middle_axis_um": middle_axis_um,
        "minor_axis_um": minor_axis_um,
        "elongation": elongation,
        "flatness": flatness,
        "extent": float(extent),
        "bbox_depth_um": float(
            bbox_dimensions_um[0]
        ),
        "bbox_height_um": float(
            bbox_dimensions_um[1]
        ),
        "bbox_width_um": float(
            bbox_dimensions_um[2]
        ),
        "centroid_z_um": float(
            centroid_physical[0]
        ),
        "centroid_y_um": float(
            centroid_physical[1]
        ),
        "centroid_x_um": float(
            centroid_physical[2]
        ),
        "principal_vectors_zyx": eigenvectors,
        "principal_variances": eigenvalues,
    }

In [ ]:
cell_descriptions = {
    "cell_1": describe_cell_mask(
        mask=cell_1_mask,
        voxel_size_zyx=VOXEL_SIZE_ZYX,
    ),
    "cell_2": describe_cell_mask(
        mask=cell_2_mask,
        voxel_size_zyx=VOXEL_SIZE_ZYX,
    ),
    "merged_cell": describe_cell_mask(
        mask=merged_mask,
        voxel_size_zyx=VOXEL_SIZE_ZYX,
    ),
}

## Spatial Bayesian merge splitting — implementation plan

This section deliberately uses only the selected component from the **merged
frame**:

- `merged_mask`
- `merged_preprocessed`
- `VOXEL_SIZE_ZYX`

The earlier parent frames remain available only for the final, after-the-fact
audit. Their masks, centroids, volumes, motion, and track identities are not
used to place markers, construct a watershed, calculate probabilities, or make
the split decision.

### Planned inference sequence

1. **Generate liberal geometric candidates.** Detect EDT maxima over several
   physical smoothing scales and H-maxima levels. At this stage a maximum is
   only a candidate geometric center, not a cell.
2. **Build pairwise EDT branch evidence.** For every candidate pair, measure the
   widest-path saddle, normalized branch persistence, branch-owned volume,
   physical separation, and peak persistence.
3. **Estimate distinct-lobe probability.** Combine those measurements with
   broad Beta likelihood models for `same lobe` and `distinct lobes`. These are
   explicit prior-informed probabilities, but they are not empirically
   calibrated until reference single-cell and merge datasets are collected.
4. **Collapse redundant maxima.** Candidate peaks with high posterior
   probability of belonging to the same lobe are grouped. Only one
   representative from each group is used as a watershed marker candidate.
5. **Evaluate `H1` and `H2` first.** Compare one-cell and two-cell hypotheses
   using conditional priors that favour the common two-cell merge, plus lobe,
   neck, child-shape, child-volume, and fragment evidence.
6. **Evaluate `H3` hierarchically.** A three-cell hypothesis is generated only
   when three independently supported lobes remain. It must then beat the
   accepted two-cell interpretation by substantially stronger posterior odds.
7. **Use broad safety hard gates only.** Hard gates reject impossible or
   numerically unsafe segmentations: markers outside the mask, missing mask
   coverage, disconnected children, extremely tiny children, or sub-resolution
   marker spacing. Normal biological variation remains probabilistic.
8. **Allow uncertainty.** If posterior evidence is insufficient, the component
   remains unsplit with status `uncertain_no_split`; the strongest rejected
   split remains visible for diagnosis.

In [ ]:
from itertools import combinations
import heapq

import matplotlib.pyplot as plt
from scipy import ndimage
from scipy.special import expit, logsumexp
from scipy.stats import beta as beta_distribution
from skimage.morphology import h_maxima
from skimage.segmentation import watershed


# ============================================================
# Spatial-only probabilistic split configuration
# ============================================================

SPLIT_CONFIG = {
    # Liberal multi-scale peak generation.
    "sigma_levels_um": (0.25, 0.40, 0.60, 0.85, 1.10),
    "h_levels_um": (0.10, 0.18, 0.28, 0.42, 0.60),
    "peak_cluster_radius_um": 1.10,
    "max_candidate_peaks": 10,
    "max_combinations_per_k": 60,

    # The merge tree uses a lightly smoothed physical EDT. The final watershed
    # uses a slightly smoother field for stable boundaries.
    "merge_tree_sigma_um": 0.20,
    "watershed_sigma_um": 0.50,

    # The current model explicitly supports the common 2-cell merge and the
    # rarer 3-cell merge. It does not generate k >= 4.
    "max_cells": 3,

    # Broad safety hard gates: only impossible/sub-resolution proposals.
    "hard_min_marker_separation_um": 0.85,
    "hard_min_child_voxels": 24,
    "hard_min_child_fraction": {2: 0.025, 3: 0.020},
    "hard_min_equivalent_radius_um": 0.55,

    # Collapse maxima very likely to belong to the same EDT lobe.
    "same_lobe_collapse_probability": 0.76,

    # Pairwise same-lobe versus distinct-lobe model.
    "pair_prior_distinct": 0.35,
    "pair_likelihood_temperature": 1.60,
    "pair_feature_models": {
        "branch_persistence": {
            "distinct": (4.0, 2.2),
            "same": (1.5, 5.0),
            "weight": 1.35,
        },
        "branch_balance": {
            "distinct": (3.0, 2.7),
            "same": (1.2, 6.0),
            "weight": 1.15,
        },
        "separation_support": {
            "distinct": (4.0, 2.2),
            "same": (2.0, 4.2),
            "weight": 0.75,
        },
        "peak_support": {
            "distinct": (3.5, 1.9),
            "same": (2.0, 2.8),
            "weight": 0.45,
        },
    },

    # Priors conditional on a component already flagged as suspicious.
    "hypothesis_priors": {1: 0.24, 2: 0.66, 3: 0.10},

    # Product-of-experts weights on smooth evidence probabilities.
    "hypothesis_evidence_weights": {
        "lobe_support": 1.35,
        "coverage_support": 2.50,
        "marker_quality": 0.35,
        "neck_support": 1.05,
        "child_shape": 0.85,
        "shape_improvement": 0.75,
        "child_volume": 0.55,
        "fragment_safety": 1.10,
    },

    # H3 is generated only with three independently supported lobes.
    "k3_generation_min_pair_probability": 0.22,
    "k3_generation_min_geometric_probability": 0.40,

    # Hierarchical posterior decision rules.
    "h2_min_conditional_probability": 0.62,
    "h2_min_odds_vs_h1": 1.65,
    "h3_min_conditional_probability": 0.78,
    "h3_min_odds_vs_h2": 3.50,
    "h1_confident_conditional_probability": 0.38,

    "probability_epsilon": 1e-6,
}

SPLIT_CONFIG

In [ ]:
# ============================================================
# Multi-scale persistent EDT peak generation
# ============================================================

def physical_sigma_voxels(
    sigma_um: float,
    voxel_size_zyx: np.ndarray,
) -> np.ndarray:
    """Convert an isotropic physical Gaussian width into voxel units."""

    voxel_size = np.asarray(voxel_size_zyx, dtype=float)

    if voxel_size.shape != (3,) or np.any(voxel_size <= 0):
        raise ValueError(
            "voxel_size_zyx must contain three positive values."
        )

    return float(sigma_um) / voxel_size


def physical_distance_between_points(
    point_a_zyx: np.ndarray,
    point_b_zyx: np.ndarray,
    voxel_size_zyx: np.ndarray,
) -> float:
    delta = (
        np.asarray(point_a_zyx, dtype=float)
        - np.asarray(point_b_zyx, dtype=float)
    )

    return float(
        np.linalg.norm(
            delta * np.asarray(voxel_size_zyx, dtype=float)
        )
    )


def _peak_position_from_record(
    peak_record: dict,
) -> tuple[int, int, int]:
    return (
        int(peak_record["z"]),
        int(peak_record["y"]),
        int(peak_record["x"]),
    )


def _representative_peak_position(
    peak_component: np.ndarray,
    smoothed_distance: np.ndarray,
) -> tuple[int, int, int]:
    coordinates = np.argwhere(peak_component)

    if len(coordinates) == 0:
        raise ValueError("Peak component is empty.")

    values = smoothed_distance[tuple(coordinates.T)]

    return tuple(
        int(value)
        for value in coordinates[int(np.argmax(values))]
    )


def _cluster_peak_detections(
    detections: list[dict],
    voxel_size_zyx: np.ndarray,
    cluster_radius_um: float,
) -> list[list[dict]]:
    """Greedily merge repeated detections of the same physical maximum."""

    ordered = sorted(
        detections,
        key=lambda record: (
            record["raw_depth_um"],
            record["smoothed_depth_um"],
        ),
        reverse=True,
    )

    clusters: list[list[dict]] = []

    for detection in ordered:
        point = np.asarray(detection["position_zyx"], dtype=float)

        best_index = None
        best_distance = np.inf

        for index, cluster in enumerate(clusters):
            representative = np.asarray(
                max(
                    cluster,
                    key=lambda record: (
                        record["raw_depth_um"],
                        record["smoothed_depth_um"],
                    ),
                )["position_zyx"],
                dtype=float,
            )

            distance = physical_distance_between_points(
                point,
                representative,
                voxel_size_zyx,
            )

            if distance < best_distance:
                best_index = index
                best_distance = distance

        if (
            best_index is not None
            and best_distance <= float(cluster_radius_um)
        ):
            clusters[best_index].append(detection)
        else:
            clusters.append([detection])

    return clusters


def detect_persistent_distance_peaks(
    mask: np.ndarray,
    voxel_size_zyx: np.ndarray,
    config: dict,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, pd.DataFrame]:
    """Return raw, merge-tree and watershed EDTs plus peak candidates."""

    mask = np.asarray(mask, dtype=bool)

    if mask.ndim != 3 or not mask.any():
        raise ValueError("mask must be a non-empty 3-D binary array.")

    voxel_size = np.asarray(voxel_size_zyx, dtype=float)

    raw_distance = ndimage.distance_transform_edt(
        mask,
        sampling=voxel_size,
    ).astype(np.float32)

    detections: list[dict] = []

    sigma_levels = tuple(
        float(value)
        for value in config["sigma_levels_um"]
    )
    h_levels = tuple(
        float(value)
        for value in config["h_levels_um"]
    )

    for sigma_um in sigma_levels:
        smoothed_distance = ndimage.gaussian_filter(
            raw_distance,
            sigma=physical_sigma_voxels(
                sigma_um,
                voxel_size,
            ),
            mode="nearest",
        )

        for h_um in h_levels:
            maxima_mask = h_maxima(
                smoothed_distance,
                h=h_um,
            ) & mask

            peak_components, component_count = ndimage.label(
                maxima_mask,
                structure=ndimage.generate_binary_structure(3, 1),
            )

            for component_id in range(1, component_count + 1):
                component = peak_components == component_id

                position = _representative_peak_position(
                    component,
                    smoothed_distance,
                )

                detections.append(
                    {
                        "position_zyx": position,
                        "sigma_um": sigma_um,
                        "h_um": h_um,
                        "raw_depth_um": float(
                            raw_distance[position]
                        ),
                        "smoothed_depth_um": float(
                            smoothed_distance[position]
                        ),
                    }
                )

    if not detections:
        global_peak = tuple(
            int(value)
            for value in np.unravel_index(
                int(np.argmax(raw_distance)),
                raw_distance.shape,
            )
        )

        detections.append(
            {
                "position_zyx": global_peak,
                "sigma_um": 0.0,
                "h_um": 0.0,
                "raw_depth_um": float(raw_distance[global_peak]),
                "smoothed_depth_um": float(raw_distance[global_peak]),
            }
        )

    clusters = _cluster_peak_detections(
        detections=detections,
        voxel_size_zyx=voxel_size,
        cluster_radius_um=float(
            config["peak_cluster_radius_um"]
        ),
    )

    maximum_distance = max(float(raw_distance.max()), 1e-6)
    peak_records: list[dict] = []

    for peak_id, cluster in enumerate(clusters, start=1):
        representative = max(
            cluster,
            key=lambda record: (
                record["raw_depth_um"],
                record["smoothed_depth_um"],
            ),
        )

        unique_sigmas = {
            float(record["sigma_um"])
            for record in cluster
        }
        unique_h = {
            float(record["h_um"])
            for record in cluster
        }
        unique_settings = {
            (
                float(record["sigma_um"]),
                float(record["h_um"]),
            )
            for record in cluster
        }

        scale_support = len(unique_sigmas) / max(len(sigma_levels), 1)
        h_support = len(unique_h) / max(len(h_levels), 1)
        setting_support = (
            len(unique_settings)
            / max(len(sigma_levels) * len(h_levels), 1)
        )

        depth_score = (
            float(representative["raw_depth_um"])
            / maximum_distance
        )

        persistence_score = float(
            0.35 * scale_support
            + 0.25 * h_support
            + 0.25 * depth_score
            + 0.15 * np.sqrt(setting_support)
        )

        position = tuple(
            int(value)
            for value in representative["position_zyx"]
        )

        peak_records.append(
            {
                "peak_id": int(peak_id),
                "z": position[0],
                "y": position[1],
                "x": position[2],
                "raw_depth_um": float(
                    representative["raw_depth_um"]
                ),
                "smoothed_depth_um": float(
                    representative["smoothed_depth_um"]
                ),
                "scale_support": float(scale_support),
                "h_support": float(h_support),
                "setting_support": float(setting_support),
                "detection_count": int(len(cluster)),
                "persistence_score": persistence_score,
            }
        )

    peak_table = (
        pd.DataFrame(peak_records)
        .sort_values(
            ["persistence_score", "raw_depth_um"],
            ascending=False,
        )
        .reset_index(drop=True)
    )

    peak_table["rank"] = np.arange(
        len(peak_table),
        dtype=int,
    ) + 1

    merge_tree_distance = ndimage.gaussian_filter(
        raw_distance,
        sigma=physical_sigma_voxels(
            float(config["merge_tree_sigma_um"]),
            voxel_size,
        ),
        mode="nearest",
    ).astype(np.float32)

    watershed_distance = ndimage.gaussian_filter(
        raw_distance,
        sigma=physical_sigma_voxels(
            float(config["watershed_sigma_um"]),
            voxel_size,
        ),
        mode="nearest",
    ).astype(np.float32)

    return (
        raw_distance,
        merge_tree_distance,
        watershed_distance,
        peak_table,
    )

In [ ]:
raw_merged_distance, merge_tree_distance, watershed_distance, peak_table = (
    detect_persistent_distance_peaks(
        mask=merged_mask,
        voxel_size_zyx=VOXEL_SIZE_ZYX,
        config=SPLIT_CONFIG,
    )
)

print("Persistent peak candidates:", len(peak_table))

display(
    peak_table[
        [
            "rank",
            "peak_id",
            "z",
            "y",
            "x",
            "raw_depth_um",
            "scale_support",
            "h_support",
            "setting_support",
            "persistence_score",
        ]
    ].round(4)
)

In [ ]:
# ============================================================
# EDT merge-tree pair evidence and same-lobe collapsing
# ============================================================

_NEIGHBOR_OFFSETS_6 = (
    (-1, 0, 0),
    (1, 0, 0),
    (0, -1, 0),
    (0, 1, 0),
    (0, 0, -1),
    (0, 0, 1),
)


def widest_path_saddle_level(
    distance_um: np.ndarray,
    mask: np.ndarray,
    start_zyx: tuple[int, int, int],
    end_zyx: tuple[int, int, int],
) -> float:
    """Maximum possible minimum EDT value along a 6-connected path."""

    distance = np.asarray(distance_um, dtype=float)
    mask = np.asarray(mask, dtype=bool)

    if not mask[start_zyx] or not mask[end_zyx]:
        raise ValueError("Both peak positions must lie inside the mask.")

    best = np.full(distance.shape, -np.inf, dtype=np.float32)
    start_value = float(distance[start_zyx])
    best[start_zyx] = start_value

    queue: list[tuple[float, tuple[int, int, int]]] = [
        (-start_value, start_zyx)
    ]

    shape = distance.shape

    while queue:
        negative_score, current = heapq.heappop(queue)
        current_score = -float(negative_score)

        if current == end_zyx:
            return current_score

        if current_score < float(best[current]) - 1e-8:
            continue

        z, y, x = current

        for dz, dy, dx in _NEIGHBOR_OFFSETS_6:
            neighbor = (z + dz, y + dy, x + dx)

            if not (
                0 <= neighbor[0] < shape[0]
                and 0 <= neighbor[1] < shape[1]
                and 0 <= neighbor[2] < shape[2]
            ):
                continue

            if not mask[neighbor]:
                continue

            candidate = min(
                current_score,
                float(distance[neighbor]),
            )

            if candidate > float(best[neighbor]) + 1e-8:
                best[neighbor] = candidate
                heapq.heappush(
                    queue,
                    (-candidate, neighbor),
                )

    raise RuntimeError(
        "Peak candidates are not connected inside the selected component."
    )


def branch_size_above_saddle(
    distance_um: np.ndarray,
    mask: np.ndarray,
    peak_zyx: tuple[int, int, int],
    saddle_um: float,
) -> int:
    """Size of the peak-owned high-EDT branch immediately above a saddle."""

    epsilon = max(
        np.finfo(np.float32).eps,
        1e-5 * max(float(distance_um[peak_zyx]), 1.0),
    )

    branch_mask = (
        np.asarray(mask, dtype=bool)
        & (np.asarray(distance_um, dtype=float) > float(saddle_um) + epsilon)
    )

    if not branch_mask[peak_zyx]:
        return 1

    labels, _ = ndimage.label(
        branch_mask,
        structure=ndimage.generate_binary_structure(3, 1),
    )

    label = int(labels[peak_zyx])

    if label <= 0:
        return 1

    return int(np.count_nonzero(labels == label))


def _safe_beta_logpdf(
    value: float,
    parameters: tuple[float, float],
    epsilon: float,
) -> float:
    clipped = float(np.clip(value, epsilon, 1.0 - epsilon))
    a, b = (float(parameters[0]), float(parameters[1]))

    return float(beta_distribution.logpdf(clipped, a, b))


def pair_distinct_lobe_probability(
    transformed_features: dict[str, float],
    config: dict,
) -> tuple[float, float]:
    """Posterior P(distinct lobes | pairwise EDT branch features)."""

    epsilon = float(config["probability_epsilon"])
    prior = float(config["pair_prior_distinct"])

    log_odds = float(
        np.log(prior + epsilon)
        - np.log(1.0 - prior + epsilon)
    )

    for feature_name, model in config[
        "pair_feature_models"
    ].items():
        value = float(transformed_features[feature_name])
        weight = float(model["weight"])

        distinct_logpdf = _safe_beta_logpdf(
            value,
            tuple(model["distinct"]),
            epsilon,
        )
        same_logpdf = _safe_beta_logpdf(
            value,
            tuple(model["same"]),
            epsilon,
        )

        log_odds += weight * (
            distinct_logpdf - same_logpdf
        )

    log_odds /= max(
        float(config["pair_likelihood_temperature"]),
        epsilon,
    )

    posterior = float(expit(log_odds))

    return posterior, float(log_odds)


def build_peak_pair_table(
    peak_table: pd.DataFrame,
    mask: np.ndarray,
    merge_tree_distance: np.ndarray,
    voxel_size_zyx: np.ndarray,
    config: dict,
) -> pd.DataFrame:
    records = peak_table.head(
        int(config["max_candidate_peaks"])
    ).to_dict("records")

    pair_records: list[dict] = []
    total_voxels = max(int(np.count_nonzero(mask)), 1)

    for first, second in combinations(records, 2):
        first_position = _peak_position_from_record(first)
        second_position = _peak_position_from_record(second)

        first_depth = float(
            merge_tree_distance[first_position]
        )
        second_depth = float(
            merge_tree_distance[second_position]
        )

        smaller_depth = max(
            min(first_depth, second_depth),
            1e-6,
        )

        saddle_um = widest_path_saddle_level(
            distance_um=merge_tree_distance,
            mask=mask,
            start_zyx=first_position,
            end_zyx=second_position,
        )

        branch_persistence = float(
            np.clip(
                1.0 - saddle_um / smaller_depth,
                0.0,
                1.0,
            )
        )

        first_branch_voxels = branch_size_above_saddle(
            merge_tree_distance,
            mask,
            first_position,
            saddle_um,
        )
        second_branch_voxels = branch_size_above_saddle(
            merge_tree_distance,
            mask,
            second_position,
            saddle_um,
        )

        smaller_branch_fraction = float(
            min(first_branch_voxels, second_branch_voxels)
            / total_voxels
        )

        # Branch cores immediately above a saddle are naturally much smaller
        # than the full component. A fraction around 8% already represents a
        # substantial independent high-EDT branch, so it maps near one.
        branch_balance = float(
            np.clip(
                smaller_branch_fraction / 0.08,
                0.0,
                1.0,
            )
        )

        separation_um = physical_distance_between_points(
            first_position,
            second_position,
            voxel_size_zyx,
        )

        separation_ratio = float(
            separation_um
            / max(first_depth + second_depth, 1e-6)
        )

        separation_support = float(
            separation_ratio / (separation_ratio + 0.65)
        )

        peak_support = float(
            np.sqrt(
                max(float(first["persistence_score"]), 0.0)
                * max(float(second["persistence_score"]), 0.0)
            )
        )

        transformed = {
            "branch_persistence": branch_persistence,
            "branch_balance": branch_balance,
            "separation_support": separation_support,
            "peak_support": peak_support,
        }

        distinct_probability, distinct_log_odds = (
            pair_distinct_lobe_probability(
                transformed,
                config,
            )
        )

        pair_records.append(
            {
                "peak_id_a": int(first["peak_id"]),
                "peak_id_b": int(second["peak_id"]),
                "separation_um": separation_um,
                "separation_ratio": separation_ratio,
                "peak_depth_a_um": first_depth,
                "peak_depth_b_um": second_depth,
                "saddle_um": float(saddle_um),
                "saddle_ratio": float(
                    saddle_um / smaller_depth
                ),
                "branch_persistence": branch_persistence,
                "branch_voxels_a": first_branch_voxels,
                "branch_voxels_b": second_branch_voxels,
                "smaller_branch_fraction": smaller_branch_fraction,
                "branch_balance": branch_balance,
                "separation_support": separation_support,
                "peak_support": peak_support,
                "distinct_lobe_log_odds": distinct_log_odds,
                "distinct_lobe_probability": distinct_probability,
                "same_lobe_probability": 1.0 - distinct_probability,
            }
        )

    if not pair_records:
        return pd.DataFrame(
            columns=[
                "peak_id_a",
                "peak_id_b",
                "distinct_lobe_probability",
                "same_lobe_probability",
            ]
        )

    return (
        pd.DataFrame(pair_records)
        .sort_values(
            "distinct_lobe_probability",
            ascending=False,
        )
        .reset_index(drop=True)
    )


def collapse_same_lobe_peaks(
    peak_table: pd.DataFrame,
    pair_table: pd.DataFrame,
    config: dict,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Complete-link grouping avoids transitive same-lobe chain collapse."""

    candidate_table = (
        peak_table
        .head(int(config["max_candidate_peaks"]))
        .copy()
        .reset_index(drop=True)
    )

    same_lookup = {
        frozenset(
            (
                int(record["peak_id_a"]),
                int(record["peak_id_b"]),
            )
        ): float(record["same_lobe_probability"])
        for record in pair_table.to_dict("records")
    }

    ordered_records = sorted(
        candidate_table.to_dict("records"),
        key=lambda record: (
            float(record["persistence_score"])
            * float(record["raw_depth_um"]),
            float(record["raw_depth_um"]),
        ),
        reverse=True,
    )

    threshold = float(
        config["same_lobe_collapse_probability"]
    )

    groups: list[list[dict]] = []

    for record in ordered_records:
        peak_id = int(record["peak_id"])
        compatible_group = None
        best_minimum_probability = -np.inf

        for group_index, group in enumerate(groups):
            probabilities = [
                same_lookup.get(
                    frozenset(
                        (
                            peak_id,
                            int(member["peak_id"]),
                        )
                    ),
                    0.0,
                )
                for member in group
            ]

            minimum_probability = (
                min(probabilities)
                if probabilities
                else 1.0
            )

            if (
                minimum_probability >= threshold
                and minimum_probability
                > best_minimum_probability
            ):
                compatible_group = group_index
                best_minimum_probability = minimum_probability

        if compatible_group is None:
            groups.append([record])
        else:
            groups[compatible_group].append(record)

    lobe_id_by_peak: dict[int, int] = {}
    representative_ids: set[int] = set()

    for lobe_id, group in enumerate(groups, start=1):
        representative = max(
            group,
            key=lambda record: (
                float(record["persistence_score"])
                * float(record["raw_depth_um"]),
                float(record["raw_depth_um"]),
            ),
        )

        representative_ids.add(
            int(representative["peak_id"])
        )

        for record in group:
            lobe_id_by_peak[
                int(record["peak_id"])
            ] = int(lobe_id)

    candidate_table["effective_lobe_id"] = (
        candidate_table["peak_id"]
        .map(lobe_id_by_peak)
        .astype(int)
    )
    candidate_table["is_effective_representative"] = (
        candidate_table["peak_id"]
        .astype(int)
        .isin(representative_ids)
    )
    candidate_table["same_lobe_cluster_size"] = (
        candidate_table["effective_lobe_id"]
        .map(
            candidate_table[
                "effective_lobe_id"
            ].value_counts()
        )
        .astype(int)
    )

    effective_table = (
        candidate_table[
            candidate_table["is_effective_representative"]
        ]
        .sort_values(
            ["persistence_score", "raw_depth_um"],
            ascending=False,
        )
        .reset_index(drop=True)
    )

    effective_table["effective_rank"] = (
        np.arange(len(effective_table), dtype=int) + 1
    )

    return candidate_table, effective_table

In [ ]:
peak_pair_table = build_peak_pair_table(
    peak_table=peak_table,
    mask=merged_mask,
    merge_tree_distance=merge_tree_distance,
    voxel_size_zyx=VOXEL_SIZE_ZYX,
    config=SPLIT_CONFIG,
)

annotated_peak_table, effective_peak_table = (
    collapse_same_lobe_peaks(
        peak_table=peak_table,
        pair_table=peak_pair_table,
        config=SPLIT_CONFIG,
    )
)

print("Raw candidate peaks:", len(annotated_peak_table))
print("Effective lobe representatives:", len(effective_peak_table))

display(
    annotated_peak_table[
        [
            "rank",
            "peak_id",
            "z",
            "y",
            "x",
            "raw_depth_um",
            "persistence_score",
            "effective_lobe_id",
            "same_lobe_cluster_size",
            "is_effective_representative",
        ]
    ].round(4)
)

if peak_pair_table.empty:
    print("Only one peak candidate was available; no pair table exists.")
else:
    display(
        peak_pair_table[
            [
                "peak_id_a",
                "peak_id_b",
                "separation_um",
                "saddle_um",
                "branch_persistence",
                "smaller_branch_fraction",
                "branch_balance",
                "distinct_lobe_probability",
                "same_lobe_probability",
            ]
        ].round(4)
    )

In [ ]:
# ============================================================
# Region, interface and broad safety measurements
# ============================================================

def region_surface_area_um2(
    region_mask: np.ndarray,
    voxel_size_zyx: np.ndarray,
) -> float:
    region = np.asarray(region_mask, dtype=bool)
    voxel_size = np.asarray(voxel_size_zyx, dtype=float)

    total_area = 0.0

    for axis in range(3):
        padded = np.pad(
            region.astype(np.int8),
            [
                (1, 1) if index == axis else (0, 0)
                for index in range(3)
            ],
            mode="constant",
        )

        transitions = np.diff(
            padded,
            axis=axis,
        ) != 0

        face_area = float(
            np.prod(np.delete(voxel_size, axis))
        )

        total_area += (
            int(np.count_nonzero(transitions))
            * face_area
        )

    return float(total_area)


def pair_interface_statistics(
    labels: np.ndarray,
    label_a: int,
    label_b: int,
    distance_um: np.ndarray,
    voxel_size_zyx: np.ndarray,
) -> dict:
    labels = np.asarray(labels)
    distance_um = np.asarray(distance_um, dtype=float)
    voxel_size = np.asarray(voxel_size_zyx, dtype=float)

    interface_values: list[np.ndarray] = []
    interface_points: list[np.ndarray] = []
    interface_area_um2 = 0.0

    for axis in range(3):
        first_slice = [slice(None)] * 3
        second_slice = [slice(None)] * 3

        first_slice[axis] = slice(0, -1)
        second_slice[axis] = slice(1, None)

        first_values = labels[tuple(first_slice)]
        second_values = labels[tuple(second_slice)]

        touching = (
            (
                (first_values == label_a)
                & (second_values == label_b)
            )
            | (
                (first_values == label_b)
                & (second_values == label_a)
            )
        )

        if not np.any(touching):
            continue

        first_coordinates = np.argwhere(touching)
        second_coordinates = first_coordinates.copy()
        second_coordinates[:, axis] += 1

        first_distance = distance_um[
            tuple(first_coordinates.T)
        ]
        second_distance = distance_um[
            tuple(second_coordinates.T)
        ]

        interface_values.append(
            np.minimum(first_distance, second_distance)
        )

        interface_points.append(
            0.5 * (
                first_coordinates.astype(float)
                + second_coordinates.astype(float)
            )
        )

        face_area = float(
            np.prod(np.delete(voxel_size, axis))
        )

        interface_area_um2 += (
            int(np.count_nonzero(touching))
            * face_area
        )

    if not interface_values:
        return {
            "contact": False,
            "interface_area_um2": 0.0,
            "distance_median_um": 0.0,
            "distance_mean_um": 0.0,
            "interface_points_zyx": np.empty((0, 3), dtype=float),
        }

    values = np.concatenate(interface_values)
    points = np.concatenate(interface_points, axis=0)

    return {
        "contact": True,
        "interface_area_um2": float(interface_area_um2),
        "distance_median_um": float(np.median(values)),
        "distance_mean_um": float(np.mean(values)),
        "interface_points_zyx": points,
    }


def describe_hypothesis_regions(
    labels: np.ndarray,
    voxel_size_zyx: np.ndarray,
) -> dict[int, dict]:
    descriptions: dict[int, dict] = {}

    for label in sorted(
        int(value)
        for value in np.unique(labels)
        if int(value) > 0
    ):
        region_mask = labels == label
        description = describe_cell_mask(
            region_mask,
            voxel_size_zyx,
        )
        description["surface_area_um2"] = (
            region_surface_area_um2(
                region_mask,
                voxel_size_zyx,
            )
        )
        descriptions[label] = description

    return descriptions


def build_watershed_hypothesis(
    mask: np.ndarray,
    watershed_distance: np.ndarray,
    selected_peaks: list[dict],
) -> np.ndarray:
    marker_image = np.zeros(
        mask.shape,
        dtype=np.int32,
    )

    for marker_label, peak_record in enumerate(
        selected_peaks,
        start=1,
    ):
        position = _peak_position_from_record(peak_record)

        if not mask[position]:
            raise ValueError(
                f"Marker {position} is outside the merged mask."
            )

        marker_image[position] = marker_label

    return watershed(
        -watershed_distance,
        markers=marker_image,
        mask=mask,
        watershed_line=False,
    ).astype(np.int32)


def hard_gate_hypothesis(
    labels: np.ndarray,
    mask: np.ndarray,
    selected_peaks: list[dict],
    voxel_size_zyx: np.ndarray,
    config: dict,
) -> tuple[bool, list[str]]:
    """Reject only impossible, disconnected or sub-resolution proposals."""

    labels = np.asarray(labels)
    mask = np.asarray(mask, dtype=bool)
    reasons: list[str] = []

    if labels.shape != mask.shape:
        reasons.append("shape_mismatch")
        return False, reasons

    if np.any(labels[~mask] != 0):
        reasons.append("labels_outside_mask")

    if np.any(labels[mask] <= 0):
        reasons.append("mask_not_fully_partitioned")

    positive_labels = [
        int(value)
        for value in np.unique(labels)
        if int(value) > 0
    ]

    expected_k = len(selected_peaks)

    if len(positive_labels) != expected_k:
        reasons.append("wrong_child_count")

    if expected_k > 1:
        minimum_separation = min(
            physical_distance_between_points(
                _peak_position_from_record(first),
                _peak_position_from_record(second),
                voxel_size_zyx,
            )
            for first, second in combinations(
                selected_peaks,
                2,
            )
        )

        if minimum_separation < float(
            config["hard_min_marker_separation_um"]
        ):
            reasons.append("sub_resolution_marker_spacing")

    total_voxels = max(int(np.count_nonzero(mask)), 1)
    minimum_fraction = float(
        config["hard_min_child_fraction"].get(
            expected_k,
            0.015,
        )
    )

    connectivity = ndimage.generate_binary_structure(3, 1)

    for label in positive_labels:
        region = labels == label
        voxel_count = int(np.count_nonzero(region))

        if voxel_count < int(config["hard_min_child_voxels"]):
            reasons.append(f"child_{label}_too_small")
            continue

        if voxel_count / total_voxels < minimum_fraction:
            reasons.append(f"child_{label}_fraction_too_small")

        _, component_count = ndimage.label(
            region,
            structure=connectivity,
        )

        if int(component_count) != 1:
            reasons.append(f"child_{label}_disconnected")

        try:
            radius_um = float(
                describe_cell_mask(
                    region,
                    voxel_size_zyx,
                )["equivalent_radius_um"]
            )
        except ValueError:
            reasons.append(f"child_{label}_not_measurable")
            continue

        if radius_um < float(
            config["hard_min_equivalent_radius_um"]
        ):
            reasons.append(f"child_{label}_sub_resolution")

    for marker_label, peak_record in enumerate(
        selected_peaks,
        start=1,
    ):
        position = _peak_position_from_record(peak_record)

        if not mask[position]:
            reasons.append(
                f"marker_{marker_label}_outside_mask"
            )

        if int(labels[position]) != marker_label:
            reasons.append(
                f"marker_{marker_label}_label_mismatch"
            )

    return len(reasons) == 0, reasons

In [ ]:
# ============================================================
# Prior-informed hypothesis probabilities
# ============================================================

def _geometric_mean_probability(
    values: list[float] | np.ndarray,
    epsilon: float,
) -> float:
    values = np.asarray(values, dtype=float)

    if values.size == 0:
        return 0.5

    values = np.clip(values, epsilon, 1.0)

    return float(np.exp(np.mean(np.log(values))))


def _pair_probability_lookup(
    pair_table: pd.DataFrame,
) -> dict[frozenset[int], float]:
    return {
        frozenset(
            (
                int(record["peak_id_a"]),
                int(record["peak_id_b"]),
            )
        ): float(record["distinct_lobe_probability"])
        for record in pair_table.to_dict("records")
    }


def selected_pair_probabilities(
    selected_peaks: list[dict],
    pair_lookup: dict[frozenset[int], float],
) -> list[float]:
    probabilities: list[float] = []

    for first, second in combinations(selected_peaks, 2):
        key = frozenset(
            (
                int(first["peak_id"]),
                int(second["peak_id"]),
            )
        )
        probabilities.append(
            float(pair_lookup.get(key, 0.0))
        )

    return probabilities


def single_cell_shape_probability(
    description: dict,
) -> float:
    """Broad probability that one region remains a plausible single cell."""

    elongation_probability = float(
        expit(
            (
                4.10 - float(description["elongation"])
            )
            / 0.80
        )
    )
    flatness_probability = float(
        expit(
            (
                3.10 - float(description["flatness"])
            )
            / 0.65
        )
    )
    extent_probability = float(
        expit(
            (
                float(description["extent"]) - 0.075
            )
            / 0.035
        )
    )

    return _geometric_mean_probability(
        [
            elongation_probability,
            flatness_probability,
            extent_probability,
        ],
        1e-6,
    )


def child_shape_evidence(
    region_descriptions: dict[int, dict],
    merged_description: dict,
    epsilon: float,
) -> tuple[float, float]:
    child_probabilities = [
        single_cell_shape_probability(description)
        for description in region_descriptions.values()
    ]

    absolute_child_shape = _geometric_mean_probability(
        child_probabilities,
        epsilon,
    )

    mean_child_elongation = float(
        np.mean(
            [
                description["elongation"]
                for description in region_descriptions.values()
            ]
        )
    )
    mean_child_extent = float(
        np.mean(
            [
                description["extent"]
                for description in region_descriptions.values()
            ]
        )
    )

    elongation_improvement = float(
        expit(
            (
                float(merged_description["elongation"])
                - mean_child_elongation
            )
            / 0.40
        )
    )
    extent_improvement = float(
        expit(
            (
                mean_child_extent
                - float(merged_description["extent"])
            )
            / 0.080
        )
    )

    shape_improvement = _geometric_mean_probability(
        [
            elongation_improvement,
            extent_improvement,
        ],
        epsilon,
    )

    return absolute_child_shape, shape_improvement


def smooth_child_volume_probability(
    volume_fractions: np.ndarray,
) -> float:
    """Broad log-ratio distribution around equal-sized children."""

    count = len(volume_fractions)

    if count <= 1:
        return 1.0

    expected_fraction = 1.0 / count

    log_ratio = np.log(
        np.clip(
            volume_fractions / expected_fraction,
            1e-8,
            None,
        )
    )

    scores = np.exp(
        -0.5 * (log_ratio / 0.85) ** 2
    )

    return float(np.mean(scores))


def score_watershed_hypothesis(
    labels: np.ndarray,
    selected_peaks: list[dict],
    effective_peaks: list[dict],
    pair_table: pd.DataFrame,
    distance_um: np.ndarray,
    merged_description: dict,
    mask: np.ndarray,
    voxel_size_zyx: np.ndarray,
    config: dict,
) -> dict:
    epsilon = float(config["probability_epsilon"])

    hard_valid, hard_reasons = hard_gate_hypothesis(
        labels=labels,
        mask=mask,
        selected_peaks=selected_peaks,
        voxel_size_zyx=voxel_size_zyx,
        config=config,
    )

    if not hard_valid:
        return {
            "hard_valid": False,
            "hard_reasons": hard_reasons,
            "k": int(len(selected_peaks)),
            "labels": labels,
            "selected_peaks": selected_peaks,
            "log_posterior_unnormalized": -np.inf,
        }

    region_descriptions = describe_hypothesis_regions(
        labels,
        voxel_size_zyx,
    )

    k = len(region_descriptions)

    child_volumes = np.asarray(
        [
            region_descriptions[label]["volume_um3"]
            for label in sorted(region_descriptions)
        ],
        dtype=float,
    )

    total_volume = max(float(child_volumes.sum()), 1e-12)
    volume_fractions = child_volumes / total_volume
    minimum_child_fraction = float(volume_fractions.min())

    soft_fraction_center = {
        1: 0.50,
        2: 0.14,
        3: 0.075,
    }.get(k, 0.05)

    fragment_safety = (
        1.0
        if k == 1
        else float(
            expit(
                (
                    minimum_child_fraction
                    - soft_fraction_center
                )
                / 0.045
            )
        )
    )

    child_volume = smooth_child_volume_probability(
        volume_fractions
    )

    absolute_child_shape, shape_improvement = (
        child_shape_evidence(
            region_descriptions,
            merged_description,
            epsilon,
        )
    )

    pair_lookup = _pair_probability_lookup(pair_table)
    selected_lobe_probabilities = (
        selected_pair_probabilities(
            selected_peaks,
            pair_lookup,
        )
    )

    selected_peak_ids = {
        int(record["peak_id"])
        for record in selected_peaks
    }

    unexplained_lobe_probabilities: list[float] = []

    for candidate in effective_peaks:
        candidate_id = int(candidate["peak_id"])

        if candidate_id in selected_peak_ids:
            continue

        support_against_selected = [
            float(
                pair_lookup.get(
                    frozenset(
                        (
                            candidate_id,
                            int(selected["peak_id"]),
                        )
                    ),
                    0.0,
                )
            )
            for selected in selected_peaks
        ]

        unexplained_lobe_probabilities.append(
            _geometric_mean_probability(
                support_against_selected,
                epsilon,
            )
        )

    maximum_unexplained_lobe_probability = (
        max(unexplained_lobe_probabilities)
        if unexplained_lobe_probabilities
        else 0.0
    )

    coverage_support = float(
        np.clip(
            1.0 - maximum_unexplained_lobe_probability,
            epsilon,
            1.0,
        )
    )

    if k == 1:
        effective_pair_probabilities = (
            selected_pair_probabilities(
                effective_peaks,
                pair_lookup,
            )
        )

        maximum_pair_probability = (
            max(effective_pair_probabilities)
            if effective_pair_probabilities
            else 0.0
        )

        lobe_support = float(
            np.clip(
                1.0 - maximum_pair_probability,
                epsilon,
                1.0,
            )
        )
        neck_support = 0.75
        shape_improvement = 0.55
    else:
        lobe_support = _geometric_mean_probability(
            selected_lobe_probabilities,
            epsilon,
        )
        neck_support = 0.0

    marker_quality = _geometric_mean_probability(
        [
            float(record["persistence_score"])
            for record in selected_peaks
        ],
        epsilon,
    )

    interface_records: list[dict] = []
    all_interface_points: list[np.ndarray] = []
    pair_neck_probabilities: list[float] = []

    if k > 1:
        surfaces = {
            label: float(
                region_descriptions[label]["surface_area_um2"]
            )
            for label in region_descriptions
        }

        marker_depth_by_label = {
            marker_label: float(
                distance_um[
                    _peak_position_from_record(peak_record)
                ]
            )
            for marker_label, peak_record in enumerate(
                selected_peaks,
                start=1,
            )
        }

        for label_a, label_b in combinations(
            sorted(region_descriptions),
            2,
        ):
            interface = pair_interface_statistics(
                labels=labels,
                label_a=label_a,
                label_b=label_b,
                distance_um=distance_um,
                voxel_size_zyx=voxel_size_zyx,
            )

            if not interface["contact"]:
                continue

            smaller_peak_depth = max(
                min(
                    marker_depth_by_label[label_a],
                    marker_depth_by_label[label_b],
                ),
                1e-6,
            )

            saddle_ratio = float(
                interface["distance_median_um"]
                / smaller_peak_depth
            )

            neck_depth_probability = float(
                expit(
                    (
                        (1.0 - saddle_ratio) - 0.22
                    )
                    / 0.10
                )
            )

            normalized_interface_area = float(
                interface["interface_area_um2"]
                / max(
                    min(
                        surfaces[label_a],
                        surfaces[label_b],
                    ),
                    1e-6,
                )
            )

            interface_probability = float(
                expit(
                    (
                        0.16 - normalized_interface_area
                    )
                    / 0.045
                )
            )

            pair_neck_probability = (
                _geometric_mean_probability(
                    [
                        neck_depth_probability,
                        interface_probability,
                    ],
                    epsilon,
                )
            )

            pair_neck_probabilities.append(
                pair_neck_probability
            )
            all_interface_points.append(
                interface["interface_points_zyx"]
            )

            interface_records.append(
                {
                    "label_a": int(label_a),
                    "label_b": int(label_b),
                    "interface_area_um2": float(
                        interface["interface_area_um2"]
                    ),
                    "normalized_interface_area": (
                        normalized_interface_area
                    ),
                    "distance_median_um": float(
                        interface["distance_median_um"]
                    ),
                    "saddle_ratio": saddle_ratio,
                    "neck_depth_probability": (
                        neck_depth_probability
                    ),
                    "interface_probability": (
                        interface_probability
                    ),
                    "pair_neck_probability": (
                        pair_neck_probability
                    ),
                }
            )

        neck_support = _geometric_mean_probability(
            pair_neck_probabilities,
            epsilon,
        )

    component_probabilities = {
        "lobe_support": float(lobe_support),
        "coverage_support": float(coverage_support),
        "marker_quality": float(marker_quality),
        "neck_support": float(neck_support),
        "child_shape": float(absolute_child_shape),
        "shape_improvement": float(shape_improvement),
        "child_volume": float(child_volume),
        "fragment_safety": float(fragment_safety),
    }

    prior = float(
        config["hypothesis_priors"].get(k, epsilon)
    )

    log_likelihood = 0.0

    for name, weight in config[
        "hypothesis_evidence_weights"
    ].items():
        probability = float(
            np.clip(
                component_probabilities[name],
                epsilon,
                1.0,
            )
        )

        log_likelihood += float(weight) * np.log(
            probability
        )

    log_posterior_unnormalized = float(
        np.log(max(prior, epsilon))
        + log_likelihood
    )

    interface_points = (
        np.concatenate(all_interface_points, axis=0)
        if all_interface_points
        else np.empty((0, 3), dtype=float)
    )

    return {
        "hard_valid": True,
        "hard_reasons": [],
        "k": int(k),
        "labels": labels,
        "selected_peaks": selected_peaks,
        "selected_pair_lobe_probabilities": (
            selected_lobe_probabilities
        ),
        "region_descriptions": region_descriptions,
        "interface_records": interface_records,
        "interface_points_zyx": interface_points,
        "minimum_child_fraction": minimum_child_fraction,
        "maximum_unexplained_lobe_probability": float(
            maximum_unexplained_lobe_probability
        ),
        "prior_probability": prior,
        "log_likelihood": float(log_likelihood),
        "log_posterior_unnormalized": (
            log_posterior_unnormalized
        ),
        **component_probabilities,
    }

In [ ]:
# ============================================================
# Hierarchical H1 -> H2 -> H3 hypothesis generation
# ============================================================

def minimum_marker_separation_um(
    selected_peaks: list[dict],
    voxel_size_zyx: np.ndarray,
) -> float:
    if len(selected_peaks) < 2:
        return np.inf

    return float(
        min(
            physical_distance_between_points(
                _peak_position_from_record(first),
                _peak_position_from_record(second),
                voxel_size_zyx,
            )
            for first, second in combinations(
                selected_peaks,
                2,
            )
        )
    )


def combination_prescore(
    selected_peaks: list[dict],
    pair_lookup: dict[frozenset[int], float],
    epsilon: float,
) -> float:
    marker_quality = _geometric_mean_probability(
        [
            float(record["persistence_score"])
            for record in selected_peaks
        ],
        epsilon,
    )

    pair_probabilities = selected_pair_probabilities(
        selected_peaks,
        pair_lookup,
    )

    lobe_support = _geometric_mean_probability(
        pair_probabilities,
        epsilon,
    )

    return float(
        0.70 * lobe_support
        + 0.30 * marker_quality
    )


def evaluate_spatial_split_hypotheses(
    mask: np.ndarray,
    distance_um: np.ndarray,
    watershed_distance: np.ndarray,
    effective_peak_table: pd.DataFrame,
    pair_table: pd.DataFrame,
    merged_description: dict,
    voxel_size_zyx: np.ndarray,
    config: dict,
) -> tuple[list[dict], list[dict]]:
    candidate_pool = (
        effective_peak_table
        .head(int(config["max_candidate_peaks"]))
        .to_dict("records")
    )

    if not candidate_pool:
        raise ValueError(
            "No effective peak candidate remains after lobe collapsing."
        )

    pair_lookup = _pair_probability_lookup(pair_table)
    epsilon = float(config["probability_epsilon"])
    all_hypotheses: list[dict] = []

    # H1: retain the original connected component.
    unsplit_labels = mask.astype(np.int32)

    all_hypotheses.append(
        score_watershed_hypothesis(
            labels=unsplit_labels,
            selected_peaks=[candidate_pool[0]],
            effective_peaks=candidate_pool,
            pair_table=pair_table,
            distance_um=distance_um,
            merged_description=merged_description,
            mask=mask,
            voxel_size_zyx=voxel_size_zyx,
            config=config,
        )
    )

    # H2 is the primary split model.
    if len(candidate_pool) >= 2:
        ranked_pairs: list[
            tuple[float, list[dict]]
        ] = []

        for pair in combinations(candidate_pool, 2):
            selected = list(pair)

            if minimum_marker_separation_um(
                selected,
                voxel_size_zyx,
            ) < float(
                config["hard_min_marker_separation_um"]
            ):
                continue

            ranked_pairs.append(
                (
                    combination_prescore(
                        selected,
                        pair_lookup,
                        epsilon,
                    ),
                    selected,
                )
            )

        ranked_pairs.sort(
            key=lambda item: item[0],
            reverse=True,
        )

        for prescore, selected in ranked_pairs[
            : int(config["max_combinations_per_k"])
        ]:
            labels = build_watershed_hypothesis(
                mask,
                watershed_distance,
                selected,
            )

            result = score_watershed_hypothesis(
                labels=labels,
                selected_peaks=selected,
                effective_peaks=candidate_pool,
                pair_table=pair_table,
                distance_um=distance_um,
                merged_description=merged_description,
                mask=mask,
                voxel_size_zyx=voxel_size_zyx,
                config=config,
            )
            result["combination_prescore"] = float(
                prescore
            )
            all_hypotheses.append(result)

    # H3 is generated only after three independent effective lobes survive.
    if (
        int(config["max_cells"]) >= 3
        and len(candidate_pool) >= 3
    ):
        ranked_triples: list[
            tuple[float, list[dict]]
        ] = []

        for triple in combinations(candidate_pool, 3):
            selected = list(triple)

            if minimum_marker_separation_um(
                selected,
                voxel_size_zyx,
            ) < float(
                config["hard_min_marker_separation_um"]
            ):
                continue

            pair_probabilities = selected_pair_probabilities(
                selected,
                pair_lookup,
            )

            if len(pair_probabilities) != 3:
                continue

            if min(pair_probabilities) < float(
                config[
                    "k3_generation_min_pair_probability"
                ]
            ):
                continue

            geometric_probability = (
                _geometric_mean_probability(
                    pair_probabilities,
                    epsilon,
                )
            )

            if geometric_probability < float(
                config[
                    "k3_generation_min_geometric_probability"
                ]
            ):
                continue

            ranked_triples.append(
                (
                    combination_prescore(
                        selected,
                        pair_lookup,
                        epsilon,
                    ),
                    selected,
                )
            )

        ranked_triples.sort(
            key=lambda item: item[0],
            reverse=True,
        )

        for prescore, selected in ranked_triples[
            : int(config["max_combinations_per_k"])
        ]:
            labels = build_watershed_hypothesis(
                mask,
                watershed_distance,
                selected,
            )

            result = score_watershed_hypothesis(
                labels=labels,
                selected_peaks=selected,
                effective_peaks=candidate_pool,
                pair_table=pair_table,
                distance_um=distance_um,
                merged_description=merged_description,
                mask=mask,
                voxel_size_zyx=voxel_size_zyx,
                config=config,
            )
            result["combination_prescore"] = float(
                prescore
            )
            all_hypotheses.append(result)

    valid_hypotheses = [
        result
        for result in all_hypotheses
        if bool(result.get("hard_valid", False))
        and np.isfinite(
            result["log_posterior_unnormalized"]
        )
    ]

    if not valid_hypotheses:
        raise RuntimeError(
            "Every spatial hypothesis failed the broad safety gates."
        )

    best_by_k: list[dict] = []

    for k in sorted(
        {
            int(result["k"])
            for result in valid_hypotheses
        }
    ):
        best_by_k.append(
            max(
                (
                    result
                    for result in valid_hypotheses
                    if int(result["k"]) == k
                ),
                key=lambda result: (
                    result["log_posterior_unnormalized"]
                ),
            )
        )

    log_values = np.asarray(
        [
            result["log_posterior_unnormalized"]
            for result in best_by_k
        ],
        dtype=float,
    )

    normalized_logs = log_values - logsumexp(log_values)

    for result, normalized_log in zip(
        best_by_k,
        normalized_logs,
    ):
        result["posterior_probability"] = float(
            np.exp(normalized_log)
        )

    return best_by_k, all_hypotheses


merged_spatial_description = describe_cell_mask(
    merged_mask,
    VOXEL_SIZE_ZYX,
)

best_hypotheses, all_evaluated_hypotheses = (
    evaluate_spatial_split_hypotheses(
        mask=merged_mask,
        distance_um=raw_merged_distance,
        watershed_distance=watershed_distance,
        effective_peak_table=effective_peak_table,
        pair_table=peak_pair_table,
        merged_description=merged_spatial_description,
        voxel_size_zyx=VOXEL_SIZE_ZYX,
        config=SPLIT_CONFIG,
    )
)

print(
    "Total generated hypotheses:",
    len(all_evaluated_hypotheses),
)
print(
    "Hard-valid hypotheses:",
    sum(
        bool(result.get("hard_valid", False))
        for result in all_evaluated_hypotheses
    ),
)

hard_rejected_records = [
    {
        "k": int(result["k"]),
        "selected_peak_ids": [
            int(record["peak_id"])
            for record in result["selected_peaks"]
        ],
        "hard_reasons": ", ".join(
            result.get("hard_reasons", [])
        ),
    }
    for result in all_evaluated_hypotheses
    if not bool(result.get("hard_valid", False))
]

if hard_rejected_records:
    print("\nBroad safety-gate rejections:")
    display(pd.DataFrame(hard_rejected_records))

In [ ]:
# ============================================================
# Hierarchical posterior decision with an explicit uncertain state
# ============================================================

hypothesis_by_k = {
    int(result["k"]): result
    for result in best_hypotheses
}

h1 = hypothesis_by_k[1]
h2 = hypothesis_by_k.get(2)
h3 = hypothesis_by_k.get(3)

epsilon = float(SPLIT_CONFIG["probability_epsilon"])

p1 = float(h1["posterior_probability"])
p2 = float(h2["posterior_probability"]) if h2 is not None else 0.0
p3 = float(h3["posterior_probability"]) if h3 is not None else 0.0

h2_conditional_probability = (
    p2 / max(p1 + p2, epsilon)
    if h2 is not None
    else 0.0
)
h2_odds_vs_h1 = (
    p2 / max(p1, epsilon)
    if h2 is not None
    else 0.0
)

h2_accepted = bool(
    h2 is not None
    and h2_conditional_probability
    >= float(
        SPLIT_CONFIG["h2_min_conditional_probability"]
    )
    and h2_odds_vs_h1
    >= float(SPLIT_CONFIG["h2_min_odds_vs_h1"])
)

if h2_accepted:
    chosen_hypothesis = h2
    decision_status = "accepted_two_cell_split"
else:
    chosen_hypothesis = h1

    if (
        h2 is not None
        and h2_conditional_probability
        > float(
            SPLIT_CONFIG[
                "h1_confident_conditional_probability"
            ]
        )
    ):
        decision_status = "uncertain_no_split"
    else:
        decision_status = "accepted_single"

h3_conditional_probability = (
    p3 / max(p2 + p3, epsilon)
    if h3 is not None and h2 is not None
    else 0.0
)
h3_odds_vs_h2 = (
    p3 / max(p2, epsilon)
    if h3 is not None and h2 is not None
    else 0.0
)

h3_accepted = bool(
    h2_accepted
    and h3 is not None
    and h3_conditional_probability
    >= float(
        SPLIT_CONFIG["h3_min_conditional_probability"]
    )
    and h3_odds_vs_h2
    >= float(SPLIT_CONFIG["h3_min_odds_vs_h2"])
)

if h3_accepted:
    chosen_hypothesis = h3
    decision_status = "accepted_three_cell_split"

split_accepted = int(chosen_hypothesis["k"]) > 1

split_candidates = [
    result
    for result in best_hypotheses
    if int(result["k"]) > 1
]

diagnostic_split_hypothesis = (
    max(
        split_candidates,
        key=lambda result: result["posterior_probability"],
    )
    if split_candidates
    else None
)

hypothesis_table = pd.DataFrame(
    [
        {
            "k": int(result["k"]),
            "posterior_probability":
                result["posterior_probability"],
            "prior_probability":
                result["prior_probability"],
            "log_likelihood":
                result["log_likelihood"],
            "lobe_support":
                result["lobe_support"],
            "coverage_support":
                result["coverage_support"],
            "unexplained_lobe_probability":
                result[
                    "maximum_unexplained_lobe_probability"
                ],
            "marker_quality":
                result["marker_quality"],
            "neck_support":
                result["neck_support"],
            "child_shape":
                result["child_shape"],
            "shape_improvement":
                result["shape_improvement"],
            "child_volume":
                result["child_volume"],
            "fragment_safety":
                result["fragment_safety"],
            "minimum_child_fraction":
                result["minimum_child_fraction"],
            "selected_peak_ids": [
                int(record["peak_id"])
                for record in result["selected_peaks"]
            ],
        }
        for result in best_hypotheses
    ]
).sort_values("k").reset_index(drop=True)

display(hypothesis_table.round(4))

decision_summary = pd.Series(
    {
        "decision_status": decision_status,
        "chosen_k": int(chosen_hypothesis["k"]),
        "split_accepted": split_accepted,
        "h2_conditional_probability":
            h2_conditional_probability,
        "h2_odds_vs_h1": h2_odds_vs_h1,
        "h3_conditional_probability":
            h3_conditional_probability,
        "h3_odds_vs_h2": h3_odds_vs_h2,
        "raw_peak_count": len(annotated_peak_table),
        "effective_lobe_count": len(effective_peak_table),
    },
    name="value",
)

display(decision_summary)

if decision_status == "uncertain_no_split":
    print(
        "The strongest split remains available for diagnosis, "
        "but the component is kept unsplit because H2 did not "
        "clear the posterior acceptance rule."
    )

In [ ]:
# ============================================================
# Inspect both the chosen output and the strongest split proposal
# ============================================================

def hypothesis_region_table(
    hypothesis: dict,
) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "label": int(label),
                "voxel_count": description["voxel_count"],
                "volume_um3": description["volume_um3"],
                "equivalent_radius_um":
                    description["equivalent_radius_um"],
                "major_axis_um":
                    description["major_axis_um"],
                "middle_axis_um":
                    description["middle_axis_um"],
                "minor_axis_um":
                    description["minor_axis_um"],
                "elongation": description["elongation"],
                "flatness": description["flatness"],
                "extent": description["extent"],
                "surface_area_um2":
                    description["surface_area_um2"],
            }
            for label, description
            in hypothesis["region_descriptions"].items()
        ]
    )


print(
    "Chosen hypothesis:",
    f"k={int(chosen_hypothesis['k'])}",
    f"({decision_status})",
)
display(
    hypothesis_region_table(
        chosen_hypothesis
    ).round(4)
)

chosen_interface_table = pd.DataFrame(
    chosen_hypothesis["interface_records"]
)

if not chosen_interface_table.empty:
    display(chosen_interface_table.round(4))


if diagnostic_split_hypothesis is not None:
    print(
        "\nStrongest split proposal:",
        f"k={int(diagnostic_split_hypothesis['k'])}",
        "posterior="
        f"{diagnostic_split_hypothesis['posterior_probability']:.4f}",
    )

    display(
        hypothesis_region_table(
            diagnostic_split_hypothesis
        ).round(4)
    )

    diagnostic_interface_table = pd.DataFrame(
        diagnostic_split_hypothesis[
            "interface_records"
        ]
    )

    if not diagnostic_interface_table.empty:
        display(diagnostic_interface_table.round(4))

In [ ]:
# ============================================================
# 2-D maximum-projection comparison
# ============================================================

hypothesis_count = len(best_hypotheses)

figure, axes = plt.subplots(
    1,
    hypothesis_count,
    figsize=(6 * hypothesis_count, 6),
    squeeze=False,
)

image_mip = merged_preprocessed.max(axis=0)

for axis, result in zip(
    axes.ravel(),
    best_hypotheses,
):
    label_mip = result["labels"].max(axis=0)

    axis.imshow(image_mip, cmap="gray")
    axis.imshow(
        np.ma.masked_where(
            label_mip == 0,
            label_mip,
        ),
        alpha=0.45,
    )

    selected_points = np.asarray(
        [
            _peak_position_from_record(record)
            for record in result["selected_peaks"]
        ],
        dtype=float,
    )

    axis.scatter(
        selected_points[:, 2],
        selected_points[:, 1],
        s=75,
        marker="x",
    )

    axis.set_title(
        "k={k} | posterior={posterior:.3f}".format(
            k=int(result["k"]),
            posterior=float(
                result["posterior_probability"]
            ),
        )
    )
    axis.set_xlabel("X voxel")
    axis.set_ylabel("Y voxel")

figure.suptitle(
    "Bayesian spatial hypotheses — merged frame only"
)
figure.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Napari inspection
# ============================================================

import napari


viewer = napari.Viewer(ndisplay=3)

viewer.add_image(
    merged_preprocessed,
    name="Merged preprocessed",
    scale=VOXEL_SIZE_ZYX,
    rendering="mip",
)

viewer.add_labels(
    merged_mask.astype(np.uint8),
    name="Original merged mask",
    scale=VOXEL_SIZE_ZYX,
    opacity=0.25,
    visible=False,
)

viewer.add_image(
    raw_merged_distance,
    name="Physical EDT (µm)",
    scale=VOXEL_SIZE_ZYX,
    opacity=0.65,
    rendering="mip",
    visible=False,
)

viewer.add_image(
    merge_tree_distance,
    name="Merge-tree EDT (µm)",
    scale=VOXEL_SIZE_ZYX,
    opacity=0.65,
    rendering="mip",
    visible=False,
)

all_peak_points = annotated_peak_table[
    ["z", "y", "x"]
].to_numpy(dtype=float)

viewer.add_points(
    all_peak_points,
    name="All persistent maxima",
    scale=VOXEL_SIZE_ZYX,
    size=2.0,
    face_color="yellow",
    opacity=0.65,
    ndim=3,
)

effective_peak_points = effective_peak_table[
    ["z", "y", "x"]
].to_numpy(dtype=float)

viewer.add_points(
    effective_peak_points,
    name="Effective lobe representatives",
    scale=VOXEL_SIZE_ZYX,
    size=3.0,
    face_color="orange",
    opacity=0.90,
    ndim=3,
)

chosen_peak_points = np.asarray(
    [
        _peak_position_from_record(record)
        for record in chosen_hypothesis["selected_peaks"]
    ],
    dtype=float,
)

viewer.add_points(
    chosen_peak_points,
    name="Chosen markers",
    scale=VOXEL_SIZE_ZYX,
    size=3.8,
    face_color="red",
    ndim=3,
)

if diagnostic_split_hypothesis is not None:
    diagnostic_peak_points = np.asarray(
        [
            _peak_position_from_record(record)
            for record
            in diagnostic_split_hypothesis["selected_peaks"]
        ],
        dtype=float,
    )

    viewer.add_points(
        diagnostic_peak_points,
        name="Strongest split markers",
        scale=VOXEL_SIZE_ZYX,
        size=3.4,
        face_color="cyan",
        ndim=3,
        visible=(
            diagnostic_split_hypothesis
            is not chosen_hypothesis
        ),
    )

for result in best_hypotheses:
    k = int(result["k"])

    viewer.add_labels(
        result["labels"].astype(np.int32),
        name=(
            f"H{k} posterior="
            f"{result['posterior_probability']:.3f}"
        ),
        scale=VOXEL_SIZE_ZYX,
        opacity=0.60,
        visible=(result is chosen_hypothesis),
    )

interface_hypothesis = (
    diagnostic_split_hypothesis
    if diagnostic_split_hypothesis is not None
    else chosen_hypothesis
)

interface_points = interface_hypothesis[
    "interface_points_zyx"
]

if len(interface_points) > 0:
    viewer.add_points(
        interface_points,
        name="Strongest split interface",
        scale=VOXEL_SIZE_ZYX,
        size=1.0,
        face_color="magenta",
        opacity=0.65,
        ndim=3,
    )

viewer.camera.angles = (45, 30, 135)
viewer.reset_view()

napari.run()

## Calibration status

The notebook now uses explicit priors, smooth feature likelihoods, pairwise
Beta distributions, posterior odds, and broad physical safety gates. The
resulting values are **prior-informed model probabilities**, not yet calibrated
frequencies.

Before this logic is moved into the production segmentation stage, the
distribution parameters and decision thresholds should be checked against:

- a large sample of stable individual cells, especially elongated and bent
  cells with more than one EDT maximum;
- confirmed two-cell merges;
- the available confirmed three-cell merges;
- boundary-truncated cells and very small cells.

The implementation keeps all parameters in `SPLIT_CONFIG` so that empirical
fitting can replace the current broad priors without changing the inference
architecture.

## Temporal reference audit

The curated case contains two earlier parent cells. The following audit is
strictly after inference. Parent information does not influence candidate
maxima, same-lobe collapsing, watershed generation, posterior probabilities, or
the final decision.

In [ ]:
# ============================================================
# Optional after-the-fact audit against the curated parent roles
# ============================================================

KNOWN_PARENT_COUNT = 2

reference_parent_volumes = np.sort(
    np.asarray(
        [
            cell_descriptions["cell_1"]["volume_um3"],
            cell_descriptions["cell_2"]["volume_um3"],
        ],
        dtype=float,
    )
)

chosen_child_volumes = np.sort(
    np.asarray(
        [
            description["volume_um3"]
            for description
            in chosen_hypothesis[
                "region_descriptions"
            ].values()
        ],
        dtype=float,
    )
)

audit_summary = pd.Series(
    {
        "known_parent_count": KNOWN_PARENT_COUNT,
        "chosen_spatial_count":
            int(chosen_hypothesis["k"]),
        "count_matches_curated_case":
            int(chosen_hypothesis["k"])
            == KNOWN_PARENT_COUNT,
        "decision_status": decision_status,
        "split_accepted": split_accepted,
        "h2_conditional_probability":
            h2_conditional_probability,
        "h2_odds_vs_h1": h2_odds_vs_h1,
        "parent_volume_sum_um3":
            float(reference_parent_volumes.sum()),
        "chosen_child_volume_sum_um3":
            float(chosen_child_volumes.sum()),
    },
    name="value",
)

display(audit_summary)

if len(chosen_child_volumes) == KNOWN_PARENT_COUNT:
    chosen_volume_audit = pd.DataFrame(
        {
            "reference_parent_volume_um3":
                reference_parent_volumes,
            "chosen_child_volume_um3":
                chosen_child_volumes,
        }
    )

    chosen_volume_audit["relative_volume_error"] = (
        np.abs(
            chosen_volume_audit["chosen_child_volume_um3"]
            - chosen_volume_audit[
                "reference_parent_volume_um3"
            ]
        )
        / chosen_volume_audit[
            "reference_parent_volume_um3"
        ]
    )

    display(chosen_volume_audit.round(4))
else:
    print(
        "Chosen-output per-child matching is skipped because "
        "the accepted output does not contain two children."
    )


if h2 is not None:
    h2_child_volumes = np.sort(
        np.asarray(
            [
                description["volume_um3"]
                for description
                in h2["region_descriptions"].values()
            ],
            dtype=float,
        )
    )

    if len(h2_child_volumes) == KNOWN_PARENT_COUNT:
        print("\nDiagnostic H2 volume comparison:")

        h2_volume_audit = pd.DataFrame(
            {
                "reference_parent_volume_um3":
                    reference_parent_volumes,
                "h2_child_volume_um3":
                    h2_child_volumes,
            }
        )

        h2_volume_audit["relative_volume_error"] = (
            np.abs(
                h2_volume_audit["h2_child_volume_um3"]
                - h2_volume_audit[
                    "reference_parent_volume_um3"
                ]
            )
            / h2_volume_audit[
                "reference_parent_volume_um3"
            ]
        )

        display(h2_volume_audit.round(4))